# Copiar imagens de uma pasta compartilhada para uma pasta própria do Drive

Para os arquivos passarem a ocupar a cota **desta** conta e poderem ser apagados na conta de origem.

**Antes:** em *Compartilhados comigo*, clique com o botão direito na pasta → *Organizar → Adicionar atalho* → Meu Drive.
Depois do "Tudo conferido", apague os arquivos **logado na conta de origem** e esvazie a lixeira dela.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, time
from pathlib import Path

#@markdown Atalho da pasta compartilhada e pasta própria de destino
ORIGEM = '/content/drive/MyDrive/prospectra/tiles_cnn_gee_05'  #@param {type:"string"}
DESTINO = '/content/drive/MyDrive/prospectra/tilesTIF_cnn_gee'  #@param {type:"string"}
PADRAO = '*.tif'  #@param {type:"string"}

origem, destino = Path(ORIGEM), Path(DESTINO)
assert origem.is_dir(), f'{origem} não existe — criou o atalho da pasta compartilhada no Meu Drive?'
destino.mkdir(parents=True, exist_ok=True)
arquivos = sorted(origem.glob(PADRAO))
assert arquivos, f'nenhum {PADRAO} em {origem}'
ja_existem = {f.name for f in destino.glob(PADRAO)}               # já no destino: não copia de novo
copiar = [f for f in arquivos if f.name not in ja_existem]
print(f'{len(arquivos)} arquivos na origem · {len(arquivos) - len(copiar)} já em {destino.name} (pulados) · '
      f'{len(copiar)} a copiar ({sum(f.stat().st_size for f in copiar) / 1e9:.1f} GB)')

for k, arq in enumerate(copiar, 1):
    alvo = destino / arq.name
    t = time.time()
    tmp = alvo.with_name(alvo.name + '.tmp')                          # arquivo pela metade não parece pronto
    shutil.copyfile(arq, tmp)
    tmp.rename(alvo)
    print(f'[{k}/{len(copiar)}] {arq.name}: {arq.stat().st_size / 1e9:.2f} GB em {time.time() - t:.0f} s')

faltam = [f.name for f in arquivos if not (destino / f.name).exists()]
diferentes = [f.name for f in arquivos
              if (destino / f.name).exists() and (destino / f.name).stat().st_size != f.stat().st_size]
if faltam:
    print(f'ATENÇÃO: {len(faltam)} faltando — rode de novo: {faltam[:5]}')
if diferentes:
    print(f'ATENÇÃO: {len(diferentes)} com tamanho diferente no destino (não foram substituídos; '
          f'confira antes de apagar na origem): {diferentes[:5]}')
if not faltam and not diferentes:
    print(f'Tudo conferido: {len(arquivos)} arquivos no destino com o mesmo tamanho. Pode apagar na conta de origem.')
